# Completamento del testo con GPT-2 — Teoria e codice commentato

Questo notebook riprende `1_TextCompletion.ipynb` e lo arricchisce con:
1. una parte di **teoria introduttiva** sui modelli di linguaggio autoregressivi e su GPT-2;
2. una **mappa di dove stiamo andando** (i passi concreti del notebook);
3. **commenti estesi** su ogni cella di codice, incluse le scelte implementative e cosa aspettarsi come output.

---

## 1. Teoria introduttiva

### 1.1 Cos'è un Language Model (LM)
Un modello di linguaggio stima la probabilità di una sequenza di token, tipicamente scomposta in modo **autoregressivo**:

$$P(w_1, w_2, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \dots, w_{t-1})$$

In pratica il modello impara a rispondere alla domanda: *"dato tutto quello che è stato scritto finora, qual è il prossimo token più probabile?"*. Il **text completion** (completamento del testo) è l'applicazione più diretta di questa capacità: si dà un prompt iniziale e si lascia che il modello generi, token dopo token, il seguito.

### 1.2 GPT-2: un Transformer decoder-only
GPT-2 (Radford et al., 2019) è un modello **Transformer decoder-only**:
- Usa solo lo **stack di decoder** dell'architettura Transformer originale (niente encoder separato, a differenza di un modello seq2seq come T5 o del Transformer di traduzione).
- Ogni layer applica **self-attention causale (masked)**: quando il modello elabora il token in posizione *t*, può "guardare" solo i token in posizione ≤ *t*. Questo è ciò che rende il modello **autoregressivo** e adatto alla generazione.
- È stato **pre-addestrato** (pretraining) su una grande quantità di testo generico (WebText) con un unico obiettivo self-supervised: **predire il prossimo token** (causal language modeling / next-token prediction). Non servono etichette umane: il testo stesso fornisce la supervisione (il token successivo è la "label").

### 1.3 Pretraining vs. utilizzo (inferenza)
- **Pretraining**: fase costosa (giorni/settimane su cluster GPU/TPU) in cui i pesi del modello vengono appresi da zero su enormi corpora testuali.
- **Inferenza / uso del modello pre-addestrato**: quello che facciamo in questo notebook. Carichiamo i pesi già appresi (`from_pretrained('gpt2')`) e li usiamo *senza aggiornarli* (nessun `backward()`, nessun optimizer) per generare testo. È il caso più semplice di riuso di un modello di pretraining: nessun fine-tuning, solo inferenza diretta ("zero-shot" generation).

### 1.4 Tokenizzazione (BPE)
GPT-2 non lavora su parole intere ma su **sub-word token**, ottenuti con **Byte-Pair Encoding (BPE)**. Il testo viene quindi convertito in una sequenza di ID interi (`tokenizer.encode`) prima di entrare nel modello, e i token generati vengono riconvertiti in testo leggibile con `tokenizer.decode`.

### 1.5 Come si genera il testo: `generate()`
Ad ogni passo il modello produce una distribuzione di probabilità (softmax sui logit) sull'intero vocabolario per il token successivo. Da questa distribuzione si può scegliere il token in diversi modi:
- **Greedy decoding** (`do_sample=False`): si prende sempre il token più probabile. Deterministico, ma tende a produrre testo ripetitivo/poco vario.
- **Sampling** (`do_sample=True`, usato in questo notebook): il token viene *campionato* dalla distribuzione di probabilità, introducendo variabilità. A parità di prompt, esecuzioni diverse possono produrre completamenti diversi.
- Altre strategie comuni (non usate qui ma utili da conoscere): **beam search**, **top-k sampling**, **top-p (nucleus) sampling**, controllo della **temperature**.

### 1.6 Perché questo è importante
Questo esempio è la base concettuale per capire applicazioni molto più complesse (chatbot, assistenti, autocompletamento di codice): tutte si riconducono, al livello più basso, a un modello autoregressivo che genera un token alla volta a partire da un contesto.

## 2. Dove stiamo andando (roadmap del notebook)

1. **Setup ambiente**: installiamo `tf-keras`, dipendenza richiesta da `transformers` per la compatibilità con alcuni backend Keras/TensorFlow (anche se qui useremo il backend PyTorch).
2. **Import delle librerie**: `torch` (per i tensori e l'esecuzione del modello) e le classi `GPT2LMHeadModel` / `GPT2Tokenizer` da `transformers`.
3. **Caricamento del modello e del tokenizer pre-addestrati** (`gpt2`, la versione più piccola, ~124M parametri).
4. **Definizione di una piccola classe `LLMApplications`**, pensata come contenitore estensibile di "applicazioni" costruite sopra un LLM (qui solo `text_completion`, ma in teoria potrebbe crescere con altre funzioni: summarization, Q&A, ecc.).
5. **Implementazione di `text_completion`**: incapsula il ciclo *encode → generate → decode*.
6. **Esecuzione su due prompt di esempio** e osservazione dei risultati, con particolare attenzione al fatto che, usando il sampling, l'output non è deterministico.

### 2.1 Setup dell'ambiente
`tf-keras` è una dipendenza opzionale di `transformers`, richiesta quando alcune parti della libreria fanno ancora riferimento all'API Keras "legacy" (separata da Keras 3 integrato in TensorFlow). Non la useremo direttamente: serve solo a evitare errori di import.

In [ ]:
# Installa tf-keras: dipendenza richiesta da transformers per compatibilità
# con l'API Keras legacy, anche se in questo notebook useremo PyTorch come backend.
! pip install tf-keras

### 2.2 Import

Nota: nel notebook originale la riga di import conteneva un errore di battitura (`import torchriusc`), che avrebbe causato un `ModuleNotFoundError` all'esecuzione, dato che serve `torch` (non `torchriusc`) per far funzionare `gpt2_tokenizer.encode(..., return_tensors='pt')` e l'esecuzione del modello. Qui è corretto in `import torch`.

In [ ]:
# torch: necessario perché il tokenizer restituisce tensori PyTorch ('pt')
# e il modello GPT2LMHeadModel di transformers, di default, è un modello PyTorch.
import torch

# GPT2LMHeadModel: GPT-2 con in testa uno strato lineare (LM head) che
#   proietta lo stato nascosto sul vocabolario, producendo i logit per il
#   token successivo.
# GPT2Tokenizer: tokenizer BPE associato a GPT-2 (converte testo <-> ID).
from transformers import GPT2LMHeadModel, GPT2Tokenizer

### 2.3 Caricamento di modello e tokenizer pre-addestrati

`from_pretrained('gpt2')` scarica (o recupera dalla cache locale) i pesi già addestrati da OpenAI e li carica in memoria. Da questo momento il modello è pronto per l'inferenza: **non verrà eseguito nessun training**, quindi non servono `loss`, `optimizer` o `backward()`.

Il warning *"unauthenticated requests to the HF Hub"* che si vede nell'esecuzione originale è solo un avviso di rate-limit: senza un token Hugging Face il download funziona comunque, ma con limiti di velocità/quantità più stringenti.

In [ ]:
# Carica GPT-2 già pre-addestrato.
# I pesi sono stati appresi durante il pre-training (next-token prediction
# su larga scala) e qui vengono solo caricati e usati, non aggiornati.
# GPT-2 è un Transformer decoder-only con self-attention causale (masked).
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

# Carica il tokenizer BPE associato a GPT-2: DEVE essere lo stesso tokenizer
# usato in fase di pretraining, altrimenti gli ID dei token non corrisponderebbero
# a ciò che il modello ha effettivamente imparato.
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

### 2.4 Una classe "contenitore" per applicazioni basate su LLM

L'idea di `LLMApplications` è avere un unico punto di accesso (`self.applications`, un dizionario nome → funzione) a cui in futuro si potrebbero aggiungere altre capacità costruite sullo stesso modello (es. `summarization`, `question_answering`, ...), senza cambiare l'interfaccia esterna. In questo notebook è usata solo `text_completion`.

In [ ]:
class LLMApplications:

    def __init__(self):
        # Associa il nome dell'applicazione alla relativa funzione.
        # Pensato per essere estensibile: si potrebbero aggiungere altre
        # voci (es. 'summarization': self.summarization) senza toccare
        # il resto della classe.
        self.applications = {
            'text_completion': self.text_completion
        }

    def text_completion(self, prompt):
        """Completa il testo utilizzando GPT-2"""

        # STEP 1 - ENCODE: trasforma il prompt (stringa) in una sequenza di
        # ID di token (tensore PyTorch, grazie a return_tensors='pt').
        # Questo è l'input che il modello si aspetta: non capisce testo
        # grezzo, ma solo ID numerici.
        inputs = gpt2_tokenizer.encode(
            prompt,
            return_tensors='pt'
        )

        # STEP 2 - GENERATE: usa il modello pre-addestrato per generare
        # nuovi token, uno alla volta, in modo autoregressivo (ogni nuovo
        # token viene rigenerato usando come contesto il prompt + i token
        # già generati finora).
        # - max_length=50: lunghezza massima della sequenza risultante
        #   (prompt incluso), misurata in token, non in parole/caratteri.
        # - num_return_sequences=1: genera un solo completamento; con un
        #   valore > 1 (e do_sample=True) si otterrebbero più varianti
        #   diverse per lo stesso prompt.
        # - do_sample=True: il prossimo token viene campionato dalla
        #   distribuzione di probabilità prodotta dal modello, invece di
        #   prendere sempre il token più probabile (greedy). Per questo
        #   l'output cambia da un'esecuzione all'altra: NON è deterministico.
        # Nota: qui non vengono aggiornati i pesi del modello (nessun
        # training): generate() esegue solo forward pass in modalità
        # inferenza.
        outputs = gpt2_model.generate(
            inputs,
            max_length=50,
            num_return_sequences=1,
            do_sample=True
        )

        # STEP 3 - DECODE: riconverte gli ID di token generati in testo
        # leggibile. skip_special_tokens=True rimuove eventuali token
        # speciali (es. end-of-text) dall'output finale.
        # outputs[0] perché generate() restituisce un batch di sequenze
        # (qui di dimensione 1, coerente con num_return_sequences=1).
        completion = gpt2_tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        return completion

### 2.5 Esecuzione su prompt di esempio

Ogni prompt viene passato a `text_completion`, che esegue il ciclo encode → generate → decode descritto sopra. Poiché `do_sample=True`, ri-eseguendo questa cella si possono ottenere completamenti diversi anche per lo stesso prompt: è un comportamento atteso, non un bug.

In [ ]:
# Crea un'istanza dell'applicazione.
llm_apps = LLMApplications()


# Esempio di completamento del testo.
print("TEXT COMPLETION:")

prompts = [
    "The future of AI is",
    "Machine learning helps us"
]

for prompt in prompts:

    # Passa il prompt a GPT-2 e ottieni il testo generato.
    completion = llm_apps.text_completion(prompt)

    print(f"   Prompt: '{prompt}'")
    print(f"   Completion: {completion}")
    print()

## 3. Cosa osservare nei risultati

- Il completamento **include il prompt originale** come prefisso: `generate()` restituisce l'intera sequenza (prompt + nuovi token), non solo la parte nuova.
- `max_length=50` è un limite sul **numero totale di token**, quindi la lunghezza in parole/caratteri del testo finale può variare.
- Con `do_sample=True` il testo generato è **stocastico**: due esecuzioni della stessa cella, con lo stesso prompt, possono produrre frasi diverse (a differenza del greedy decoding, che sarebbe deterministico).
- Il modello **non ha memoria persistente** tra un prompt e l'altro: ogni chiamata a `text_completion` parte da zero, condizionata solo dal prompt che le viene passato in quel momento.
- Nessuna fase di training è avvenuta: la qualità del testo generato dipende interamente da ciò che GPT-2 ha imparato durante il **pretraining**.